In [3]:
%cd ../
# Windows guard: import scikit-learn before lightning/torch to avoid a native-lib
# heap-corruption crash (silent kernel death). One line; the rest is unchanged.
import sklearn.cluster  # noqa: F401

# Import required libraries
from pathlib import Path
import hydra
from omegaconf import DictConfig, OmegaConf, open_dict
from hydra.core.global_hydra import GlobalHydra
import torch
import json, time, shutil, logging, gc, os
from lightning import seed_everything

from modules.training.training import lightning_train_model as train_model
from modules.loader.loaders import trainScaler
from modules.utils.count_params import count_parameters
from modules.utils.partialload import load_partial_state_dict
from modules.utils.hydraqol import run_decorator

d:\DSME\Code\Repos\SPP


In [4]:
#@hydra.main(config_path="../data/config", config_name="train_model", version_base="1.3")
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()
hydra.initialize(version_base=None, config_path="../data/config", job_name="test_app")

hydra.initialize()

In [6]:
cfg = hydra.compose(config_name="train_model.yaml", overrides=["training=adamw_r_patient", "path=applewood_local", "model=InceptionTime10", "dataset=SPPs_combined",
                                                            "transforms=tscHFNoise", "loader=M", "seed=0"])

In [7]:
##############################
# Preliminaries
##############################
logger = logging.getLogger("training")
save_dir = Path(cfg.save_dir)

# Set the random seed for reproducibility
seed_everything(cfg.seed)
#torch.set_flush_denormal(True) # To avoid significant performance drop on CPU when the numbers get close to 0 (like 10e-324)

# Select the device for computation (CUDA, MPS, or CPU)
device = "cuda" if (cfg.training.device=="cuda" and torch.cuda.is_available()) else "cpu"

# Train scaler if necessary
if cfg.transforms.standardize:
    cfg.dataset.mean, cfg.dataset.std = trainScaler(cfg.dataset, cfg.loader)
cfg.dataset.mean, cfg.dataset.std

Seed set to 0
d:\DSME\Code\Repos\SPP\.pixi\envs\default\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


({'gegenhalter': 64274.7266161315, 'weg_100mm': 9.596954260866548, 'weg_25mm': 5.7943427050681695, 'stempel_combined': 179612.5961829205, 'schneidkraft': 115337.86955699445, 'niederhalter_combined': 171683.52459779396, 'RE_linear': 0.4341117124620731, 'DC': 0.3012772445002695, 'ChA': 35.292997122104985, 'ChH': 0.5004001644272926},
 {'gegenhalter': 82492.57585646599, 'weg_100mm': 18.046637474176233, 'weg_25mm': 9.937632938396835, 'stempel_combined': 255545.52298611554, 'schneidkraft': 183868.3024614184, 'niederhalter_combined': 161667.52248245568, 'RE_linear': 0.2690990496764433, 'DC': 0.06239117582804819, 'ChA': 5.150251322754501, 'ChH': 0.10913015311697188})

In [ ]:
# Instantiate transforms for training and testing
pretransform = hydra.utils.instantiate(cfg.dataset.pretransform)
train_transform = hydra.utils.instantiate(cfg.transforms.train)
valid_transform = hydra.utils.instantiate(cfg.transforms.val)
test_transform = hydra.utils.instantiate(cfg.transforms.test)

# Instantiate datasets with transforms
train_dataset = hydra.utils.instantiate(cfg.dataset.train, transform=train_transform, pretransform=pretransform)
valid_dataset = hydra.utils.instantiate(cfg.dataset.val, transform=valid_transform, pretransform=pretransform)
test_dataset = hydra.utils.instantiate(cfg.dataset.test, transform=test_transform, pretransform=pretransform)

# Instantiate data loaders
train_loader, valid_loader, test_loader =  hydra.utils.instantiate(cfg.loader.object, dataset_train=train_dataset, dataset_val=valid_dataset, dataset_test=test_dataset, task=cfg.dataset.task) 

d:\DSME\Code\Repos\SPP\.pixi\envs\default\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
batch = next(iter(valid_loader))

In [ ]:
batch['y'].shape

torch.Size([256, 4])

In [ ]:
logger.info(f"Dataset splits, train: {len(train_dataset)}, valid: {len(valid_dataset)}, test: {len(test_dataset)}")

In [ ]:
# Instantiate model
model = hydra.utils.instantiate(cfg.model.object) #.to(device)
n_params = count_parameters(model)
logger.info(f"Training: {cfg.name} (params: {n_params:,})".replace(",", " "))

In [ ]:
# Load Pretrained Checkpoint (if exists)
if (cfg.pretrain_checkpoint is not None) and Path(cfg.pretrain_checkpoint).exists():
    mismatch_over_union_layers, mismatch_over_union_params = load_partial_state_dict(model, cfg.pretrain_checkpoint)
    logger.info(f"Loaded pre-trained model from {cfg.pretrain_checkpoint}: {1.0 - mismatch_over_union_layers:.2f} -- {1.0 - mismatch_over_union_params:.2f}")

In [ ]:
    ##############################
    # Actual Task: Training Loop
    ##############################
    # Training loop 
results = train_model(model, cfg, train_loader, valid_loader, test_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A500 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
d:\DSME\Code\Repos\SPP\.pixi\envs\default\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:654: Checkpoint directory D:\DSME\Code\Repos\SPP\data\models\SPP5s_combined_InceptionTime10_tsHFbasic_M_adamw__0_ exists and is not empty.
Restoring states from the checkpoint path at data\models\SPP5s_combined_InceptionTime10_tsHFbasic_M_adamw__0_\last.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name       | Type                        | Params | Mode 
--------------------------------------------

Sanity Checking DataLoader 0:  50%|█████     | 1/2 [00:00<00:00,  2.08it/s]

In [ ]:
##############################
# Saving Results
##############################
if int(os.environ.get("SLURM_PROCID", 0)) == 0:
    # Save test metrics
    with open(save_dir / "training_results.json", "w") as f:
        json.dump(results, f, indent=4)

    # Empty CUDA cache if GPU is available
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # Perform garbage collection
    gc.collect()

print( results["last"]["test_loss"])